# Blocks 1–5 — live tree from GitHub

Opening a notebook from GitHub in Colab **does not** clone `src/`. This cell pulls branch **`block1`** (the live `src/med_doc` pipeline).

The older `block1/`, `block2/`, `block3/` folders are Colab snapshots. Prefer this notebook.

**Do not upload clinic PHI to Colab.**

In [ ]:
# Pull the live repo into this Colab runtime (branch block1).
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/RwaRwa599/epq3.git"
BRANCH = "block1"
DEST = Path("/content/epq3")

def _run(cmd):
    print("$", " ".join(cmd))
    subprocess.check_call(cmd)

if not (DEST / ".git").is_dir():
    _run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO, str(DEST)])
else:
    _run(["git", "-C", str(DEST), "fetch", "origin", BRANCH])
    _run(["git", "-C", str(DEST), "checkout", BRANCH])
    _run(["git", "-C", str(DEST), "pull", "--ff-only", "origin", BRANCH])

os.chdir(DEST)
_run([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
print("cwd:", os.getcwd())
print("HEAD:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

If the repo is **private**, replace the clone URL with a token (Colab secret `GITHUB_TOKEN`):

```python
from google.colab import userdata
token = userdata.get("GITHUB_TOKEN")
REPO = f"https://{token}@github.com/RwaRwa599/epq3.git"
```

In [ ]:
# Synthetic blank → Blocks 1–5 (no PHI).
from pathlib import Path

from med_doc.htr.batch import process_from_block1
from med_doc.kg import KnowledgeGraph
from med_doc.normalization.batch import normalize_batch
from med_doc.paths import SYNTHETIC_DIR
from med_doc.rescoring import process_from_block3
from med_doc.review import process_from_block4

sheet = SYNTHETIC_DIR / "lab_request_v0_blank.png"
assert sheet.exists(), sheet

out = Path("/content/pipeline")
out.mkdir(parents=True, exist_ok=True)
kg = KnowledgeGraph.load()

b1 = normalize_batch([sheet], output_zip=out / "block1.zip")
b3 = process_from_block1(out / "block1.zip", output_zip=out / "block3.zip", kg=kg, backend="lexicon", mode="both")
b4 = process_from_block3(out / "block3.zip", output_zip=out / "block4.zip", kg=kg)
b5 = process_from_block4(out / "block4.zip", output_zip=out / "block5.zip", kg=kg)

print("Block 1 docs:", b1["manifest"]["successful_documents"])
print("Block 3:", b3["manifest"]["total_documents"], "hitl", b3["manifest"]["hitl_documents"])
print("Block 4:", b4["manifest"]["documents"][0]["n_ticked"], "ticks")
print("Block 5 needs_review:", b5["manifest"]["documents"][0]["needs_review"])
print("ZIPs:", list(out.glob("*.zip")))